# CS224N · Lecture 15 — Life after DPO (guest lecture)

**Slides:** [cs224n-spr2024-lecture15-life-after-dpo-lambert.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture15-life-after-dpo-lambert.pdf) (Nathan Lambert, Allen Institute for AI, May 21, 2024)  
**Prerequisite:** L10 (instruction tuning, reward models, RLHF, DPO). This lecture is about **practice**: how open models were actually post-trained after DPO appeared, how to evaluate reward models, and what still separates open recipes from industry.

| § | Topic | You will implement / measure |
|---|---|---|
| 1 | A brief history; why RLHF matters | — |
| 2 | Definitions; instruction fine-tuning with chat templates | A chat template with loss masking |
| 3 | The path to DPO models (Alpaca → Zephyr → Tulu 2) | — |
| 4 | **RewardBench**: evaluating reward models | Per-category RM accuracy; DPO models as reward models, with and without the reference model; the label-noise ceiling; margin losses |
| 5 | Fine-tuning a "good" model: PPO vs. DPO (Ivison et al. 2024) | — |
| 6 | **Online vs. offline preference data** | Offline DPO vs. iterative/online DPO |
| 7 | Rejection sampling (best-of-n) and the Llama 3 recipe | Best-of-n, its KL formula, and over-optimization |
| 8 | 🔎 DPO variants (IPO, KTO, ORPO, SimPO) and life after *this* lecture: RL with verifiable rewards, GRPO | Loss functions; GRPO on a verifiable task |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, collections, itertools

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))
def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)
def kl(p, q):
    return float((p * (np.log(p + 1e-12) - np.log(q + 1e-12))).sum(-1).mean())

---
## 1. A heavily abbreviated history of language models

| Year | Event |
|---|---|
| 1948 | Claude Shannon models English |
| 1948–2017 | (n-grams, neural LMs, word vectors, RNNs: L1–L7) |
| 2017 | The Transformer is born |
| 2018 | GPT-1, ELMo, and BERT released |
| 2019 | GPT-2 and scaling laws |
| 2020 | GPT-3: surprising capabilities, and many harms |
| 2021 | *Stochastic Parrots* (Bender, Gebru et al.) |
| 2022 | ChatGPT |

**Can ChatGPT exist without RLHF? RLHF seems to be necessary, but not sufficient.** It's a key factor in many popular models, on and off the record: ChatGPT, Bard/Gemini, Claude (with Constitutional AI, Bai et al. 2023), and Llama 2. The Llama 2 paper: *"Meanwhile reinforcement learning, known for its instability, seemed a somewhat shadowy field for those in the NLP research community. However, reinforcement learning proved highly effective, particularly given its cost and time effectiveness."*

---
## 2. Background: definitions, IFT, DPO, and the RLHF objective

Some definitions for "alignment" of models:
* **Instruction fine-tuning (IFT):** training a model to follow user instructions (usually via the autoregressive LM loss).
* **Supervised fine-tuning (SFT):** training a model to learn task-specific capabilities (usually via the autoregressive LM loss).
* **Alignment:** the general notion of training a model to mirror user desires, with any loss function.
* **Reinforcement learning from human feedback (RLHF):** a specific technical tool for training ML models from human data.
* **Preference fine-tuning:** using labeled preference data to fine-tune an LM (with RL, DPO, or another loss; there's also learning to rank).

### 2.1 Instruction fine-tuning
Start from a **base language model**, and continue training the Transformer on pairs of question: answer. This (1) adapts the base model to a specific **style of input**, and (2) adds the ability to include **system prompts, multi-turn dialogues, and other chat templates**, marked with **special tokens**:
```
<|system|>
You're a helpful agent
<|end|>
<|user|>
{query}
<|end|>
<|assistant|>{Answer goes here}
```
In code, rendering a conversation and masking the loss to the assistant's turns (L10 §3.3, now multi-turn):

In [ ]:
def render_chat(messages):
    # Returns token list and a loss mask: 1 only on assistant content (and its end marker).
    tokens, mask = [], []
    for m in messages:
        header = [f"<|{m['role']}|>"]; body = m["content"].split() + ["<|end|>"]
        tokens += header + body
        mask += [0] * len(header) + ([1] * len(body) if m["role"] == "assistant" else [0] * len(body))
    return tokens, mask

chat = [{"role": "system", "content": "You're a helpful agent"},
        {"role": "user", "content": "What makes a transformer a transformer?"},
        {"role": "assistant", "content": "Self-attention layers, plus feed-forward layers with residual connections."},
        {"role": "user", "content": "Thanks!"},
        {"role": "assistant", "content": "You're welcome."}]
toks, mask = render_chat(chat)
print(" ".join(t + ("*" if m else "") for t, m in zip(toks, mask)))
print(f"\n{sum(mask)} of {len(toks)} tokens are trained on (* marks them)")

🔎 **A practical trap:** the chat template used at inference must **exactly match** the one used in fine-tuning (same special tokens, same whitespace). A mismatch silently degrades quality. Hugging Face tokenizers ship the template with the model (`tokenizer.apply_chat_template(messages)`); use it rather than formatting strings by hand.

### 2.2 Review: the RLHF objective, reward modeling, and DPO
$$\max_\pi\ \mathbb{E}_{x\sim D,\,y\sim\pi(y\mid x)}\big[r_\theta(x, y)\big] - \beta\,\mathcal{D}_{KL}\big(\pi(y\mid x)\,\|\,\pi_{\text{ref}}(y\mid x)\big)$$
Optimize a "reward" inspired by human preferences, and **constrain the model not to trust the reward too much** (preferences are hard to model). The primary questions: **(1) how to implement the reward $r(x, y)$, and (2) how to optimize it.**

**Preference (reward) modeling** uses the Bradley–Terry model, estimating the probability that a pairwise preference is true: $p^*(y_1\succ y_2\mid x) = \frac{\exp r^*(x, y_1)}{\exp r^*(x, y_1) + \exp r^*(x, y_2)}$. **Can we just use supervised learning on scores?** Assigning a scalar reward of how good a response is **did not work**; pairwise preferences are easy to collect and **worked**!

**What if we just use gradient ascent on this equation?** The answer, with some math, is **Direct Preference Optimization** (Rafailov, Sharma, Mitchell et al., released May 29, 2023). DPO's characteristics:
1. **Extremely simple to implement.**
2. **Scales nicely** with existing distributed training libraries.
3. **Trains an implicit reward function** (which can still be used as a reward model; see RewardBench, §4).

The first two points mean we'll see more DPO models than anything else, and learn its limits! **DPO vs. RL (PPO, REINFORCE, …):** they're very different optimizers: DPO learns directly from preferences rather than using RL update rules. (It's also not really "offline vs. online RL", but that's more muddled.)

---
## 3. The path to DPO models

**First open instruction-tuned models (spring 2023):**

| Model | Date | Data / notes | MT-Bench |
|---|---|---|---|
| **Alpaca** | Mar 13 | 52K self-instruct-style data distilled from text-davinci-003; weight diff to LLaMA 7B | – |
| **Vicuna** | Mar 30 | ChatGPT conversations from **ShareGPT**; LLaMA 7B/13B; introduced **LLM-as-a-judge** | 6.69 (7B) |
| **Koala** | Apr 3 | Diverse data (Alpaca, Anthropic HH, ShareGPT, WebGPT…); human evaluation | 6.08 (13B) |
| **Dolly** | Apr 12 | 15K **human-written** examples; Pythia 12B | 3.28 (12B) |

* **ShareGPT data:** conversations from a tool for sharing ChatGPT chats. A **legal gray area**: most of these datasets are unlicensed/without consent. Used extensively, and starting to be replaced by carefully collected counterparts: **LMSYS-Chat-1M** (cleaned Chatbot Arena conversations) and **WildChat** (free ChatGPT usage in exchange for data).
* **OpenAssistant (OASST1, Apr 15, 2023): the first open, human instruction dataset.** 161,443 messages in 35 languages, annotated with 461,292 quality ratings, over 10,000 fully annotated conversation trees, from more than 13,500 volunteers. Still the only human dataset of this size released.
* **StableVicuna (Apr 2023): the first RLHF model** (open): trained with **PPO** on OASST1 (SFT + PPO), Anthropic HH, and Stanford Human Preferences (SHP). Standard formulation, ahead of its time.
* **Llama 2 chat backlash:** should chat models be "safe"? Llama 2 chat refused many harmless requests (XSTest, Röttger et al. 2023, measures such **exaggerated safety**). "**Uncensored**" models followed: remove instances of "as a language model" or "Sorry, …" from training data so the model doesn't refuse. (Confusing name: the models were never explicitly censored; Lambert prefers "direct" or "unbiased".)
* **Transition period** (Apr–Oct 2023): WizardLM (EvolInstruct synthetic data), UltraLM (UltraChat), OpenChat (filtered ShareGPT), XwinLM (first model to beat GPT-4 on AlpacaEval, "trained with RLHF" but no details), OpenHermes on Mistral 7B (strong synthetic data filtering + a better base model). Strong models, but none markedly shifted the narrative.

**DPO works: Zephyr β** (HuggingFace H4). The first model to make a splash with DPO: a fine-tune of **Mistral 7B** on the **UltraFeedback** dataset. It discovered the **weirdly low learning rates** that are now standard for DPO (**~5e-7**). MT-Bench 7.34.

**DPO scales: Tulu 2** (AI2). The first model to scale DPO to **70 billion parameters**, strongly validating the Zephyr results, and starting the **DPO vs. PPO debate** for real. MT-Bench 7.89 (70B).

**RLHF phase: SteerLM & Starling.** Still plenty of models showing that PPO (and RL methods) outperform DPO. SteerLM: attribute-conditioned fine-tuning. **Starling:** introduced a new preference dataset, **Nectar**, and a **k-wise reward-model loss** (moving beyond pairwise preferences). MT-Bench 8.09 (7B), beating every model except GPT-4 at the time.

**Life after DPO:** it's much easier to get into alignment research, but academics still don't really have the resources (e.g., human data) to do RLHF like industry. Two directions: **(1) better evaluation for alignment** (RewardBench; suites like Arena-Hard), and **(2) improving upon DPO models** (PPO vs. DPO studies; online DPO variants).

---
## 4. RewardBench: evaluating reward models (Lambert et al. 2024)

**From environment to reward model:** a reward model is a Transformer (usually initialized from an LM) that outputs **one scalar** per (prompt + completion). It's trained on input pairs (selected prompt+completion, rejected prompt+completion) with a loss that **increases the difference of the predicted rewards**.

**Advanced considerations:**
* Usually trained for **1 epoch** (they overfit).
* Evaluation often shows only **65–75% agreement**: preference data is noisy.
* Additional options, e.g., a **margin** between choices in the loss function (Llama 2: a larger margin when annotators said "significantly better").

**How do we evaluate reward models?** Many researchers, engineers, and papers from industry say reward models are crucial to RLHF, but there was no standard evaluation. **RewardBench** measures, for curated (prompt, chosen, rejected) triples, how often the RM scores the chosen response higher, reported by category:
* **Chat** (easy chat comparisons), **Chat Hard** (subtle changes of topic, or trick questions made intentionally), **Safety** (should refuse dangerous requests, but **not** refuse safe ones: XSTest / Do-Not-Answer), **Reasoning** (code and math with subtle bugs).

*A Chat Hard example (Zeng et al. 2023, LLMBar).* Prompt: *"Give an example of a metaphor that uses the following object: Stars."* Chosen: *"The stars were twinkling diamonds in the night sky."* Rejected: *"Her smile was as radiant as the full moon on a clear summer night."* The rejected one is fluent and is a metaphor, but doesn't use the given object.

**Findings (as of May 2024):** the leaderboard grew from 5 to 30+ models; DPO models' progress was slowing; **LLM-as-a-judge was not state of the art** as a reward model; and **Chat Hard was the only meaningful eval** (others were near saturation). Cohere's RMs led (e.g., Chat 96.4, Chat Hard 71.3, Safety 92.7, Reasoning 97.7 in May 2024). For **safety**, a good RM handles both directions; poor ones either **refuse everything** or **respond to everything**.

### 4.1 A miniature RewardBench
Synthetic responses have a true **quality**, a **length**, and whether they're a **refusal**. Each category's pairs test something different, and we evaluate several reward models:

In [ ]:
r = np.random.default_rng(0)
def make_category(n, kind):
    # Each pair: (features_chosen, features_rejected); features = [quality, length, refusal, should_refuse]
    pairs = []
    for _ in range(n):
        if kind == "Chat":                                   # big quality gap
            qc, qr = r.normal(1.5, 0.5), r.normal(-1.0, 0.5); lc, lr_ = r.normal(0, 1), r.normal(0, 1); pairs.append(([qc, lc, 0, 0], [qr, lr_, 0, 0]))
        elif kind == "Chat Hard":                            # small quality gap, rejected is LONGER and more polished-looking
            qr = r.normal(0.0, 0.3); qc = qr + r.uniform(0.05, 0.4); pairs.append(([qc, r.normal(-0.5, 0.5), 0, 0], [qr, r.normal(1.0, 0.5), 0, 0]))
        elif kind == "Chat (detailed wins)":                 # the better answer is the LONGER, more detailed one
            qr = r.normal(0.0, 0.5); qc = qr + r.uniform(0.2, 1.0); pairs.append(([qc, r.normal(1.0, 0.5), 0, 0], [qr, r.normal(-0.5, 0.5), 0, 0]))
        elif kind == "Safety (should refuse)":
            pairs.append(([0.0, r.normal(-1, 0.3), 1, 1], [r.normal(0.5, 0.5), r.normal(0.5, 0.5), 0, 1]))
        elif kind == "Safety (should comply)":
            pairs.append(([r.normal(0.5, 0.5), r.normal(0.3, 0.5), 0, 0], [0.0, r.normal(-1, 0.3), 1, 0]))
    return np.array(pairs, float)

categories = {k: make_category(300, k) for k in ["Chat", "Chat Hard", "Chat (detailed wins)", "Safety (should refuse)", "Safety (should comply)"]}

reward_models = {
    "good RM": lambda f: f[..., 0] + 2.0 * f[..., 2] * (2 * f[..., 3] - 1),              # quality, + refuses exactly when it should
    "length-biased RM": lambda f: f[..., 0] + 0.8 * f[..., 1] + 2.0 * f[..., 2] * (2 * f[..., 3] - 1),
    "refuses everything": lambda f: f[..., 0] + 3.0 * f[..., 2],
    "never refuses": lambda f: f[..., 0] - 3.0 * f[..., 2],
}
print(f"{'reward model':20s}" + "".join(f"{c:>23s}" for c in categories))
for name, rm in reward_models.items():
    accs = [np.mean(rm(P[:, 0]) > rm(P[:, 1])) for P in categories.values()]
    print(f"{name:20s}" + "".join(f"{a:23.1%}" for a in accs))

Each failure mode shows up in specific columns. The length-biased RM looks fine on easy chat, and even *better* than fine when the detailed answer wins, but fails **Chat Hard** (where the rejected response is longer) and partly fails the should-refuse category (refusals are short). The two safety extremes each fail exactly one safety direction. That's why RewardBench reports **categories**, not one average: an average can hide exactly the failure you care about.

### 4.2 Using DPO models as reward models, with and without the reference model
A DPO-trained policy defines an **implicit reward** (L10 §6): $r(x, y) = \beta\log\frac{\pi(y\mid x)}{\pi_{\text{ref}}(y\mid x)} + \beta\log Z(x)$. Since $Z(x)$ cancels in comparisons, RewardBench can score DPO models directly. **But this needs the reference model** (two forward passes). **What if we drop it** and use $\beta\log\pi(y\mid x)$ alone? Then the score is contaminated by whatever the reference model already preferred, e.g., **shorter responses** have higher sequence log-probability (fewer tokens to multiply). We simulate a DPO policy that has learned the true reward exactly:

In [ ]:
beta = 0.1
def ref_logprob(f):                         # reference model: sequence log-prob falls with length (more tokens)
    return -5.0 * (2.5 + f[..., 1]) + 0.5 * f[..., 0]
def dpo_logprob(f):                         # a DPO policy at the optimum: log pi = log pi_ref + r*/beta + const
    true_r = reward_models["good RM"](f)
    return ref_logprob(f) + true_r / beta

implicit_with_ref = lambda f: beta * (dpo_logprob(f) - ref_logprob(f))
implicit_no_ref = lambda f: beta * dpo_logprob(f)
for name, rm in [("DPO implicit reward (with reference)", implicit_with_ref), ("DPO log-prob only (no reference)", implicit_no_ref)]:
    accs = [np.mean(rm(P[:, 0]) > rm(P[:, 1])) for P in categories.values()]
    print(f"{name:38s}" + "".join(f"{a:10.1%}" for a in accs))
print(" " * 38 + "".join(f"{c[:9]:>10s}" for c in categories))
print("(columns: Chat, Chat Hard, Chat (detailed wins), Safety-refuse, Safety-comply)")

With the reference model, a perfectly trained DPO policy is a perfect reward model. Without it, the reference model's preference for **short** sequences leaks into the scores. Accuracy collapses in the category where the better answer is longer, and stays high in the categories where the better answer happens to be shorter (Chat Hard, refusals), but only because the bias points the right way there by accident. **A benchmark score can rise for the wrong reason; check per-category results against a known bias** (here, length). (The magnitude depends on $\beta$ and on how strongly $\pi_{\text{ref}}$'s log-prob depends on length; real DPO models aren't at the optimum, which adds more error.)

### 4.3 The label-noise ceiling: "evaluation often only has 65–75% agreement"
If human labels are noisy, even a **perfect** reward model can't agree with them more often than the labelers agree with the truth. And **measured** RM accuracy against noisy test labels understates its true accuracy:

In [ ]:
r = np.random.default_rng(1)
n = 5000
gap = r.normal(0, 1, n)                                          # true reward difference between two responses
true_pref = gap > 0
for flip in [0.0, 0.15, 0.30]:
    labels = np.where(r.random(n) < flip, ~true_pref, true_pref)  # human labels with a flip rate
    for rm_name, rm_gap in [("perfect RM", gap), ("decent RM", gap + r.normal(0, 0.7, n))]:
        print(f"label flip rate {flip:.0%}: {rm_name:11s} accuracy vs. noisy labels {np.mean((rm_gap > 0) == labels):.1%}   "
              f"vs. the truth {np.mean((rm_gap > 0) == true_pref):.1%}")

With a 30% flip rate (roughly the 65–75% agreement the slides mention), a *perfect* reward model scores only ~70% on held-out human labels. So when RM accuracies cluster around 70%, much of the gap to 100% may be **label noise, not model error**. (Rule of thumb: measured accuracy $\approx (1-\epsilon)\cdot a + \epsilon\cdot(1-a)$ for true accuracy $a$ and flip rate $\epsilon$.)

### 4.4 A margin in the loss
Llama 2's reward model adds a margin $m(r)$ that depends on how strongly the annotator preferred the chosen response: $\mathcal{L} = -\log\sigma\big(r(x, y_w) - r(x, y_l) - m(r)\big)$. Pairs labeled "significantly better" must be separated by more than "slightly better" ones.

In [ ]:
def bt_margin_loss_grad(score_w, score_l, margin):
    u = score_w - score_l - margin
    return -np.log(sigmoid(u)), -(1 - sigmoid(u))          # loss, d loss / d (score_w - score_l)
for gap_, m in [(1.0, 0.0), (1.0, 1.0), (3.0, 1.0)]:
    l, g = bt_margin_loss_grad(gap_, 0.0, m)
    print(f"score gap {gap_}, margin {m}: loss {l:.3f}, gradient {g:+.3f}  "
          f"{'(margin makes a 1.0 gap insufficient)' if m == 1.0 and gap_ == 1.0 else ''}")

**Towards RewardBench 2.0:** reasoning was too easy to game via formatting (bugs are small, human vs. model text is distinguishable); move from pairwise to **batch ranking** (lower random baseline); more datasets (jailbreaking, held-out custom data); more closed models; and, most importantly, **correlating RM scores with downstream PPO training results**.

---
## 5. Fine-tuning a "good" model: does PPO beat DPO? (Ivison et al. 2024)

*Unpacking DPO and PPO: Disentangling Best Practices for Learning from Preference Feedback.* Starting point: **Tulu 2 13B SFT** (Llama 2 base + a large, diverse SFT dataset), evaluated on factuality (MMLU), reasoning (GSM8k, BBH), coding (HumanEval+, MBPP+), chat (AlpacaEval 1&2, IFEval), safety (ToxiGen, XSTest), and truthfulness (TruthfulQA). Then, step by step (the slides' data were marked "not final"):

| Step | What happened |
|---|---|
| + DPO on **Anthropic HH** (all-human, noisy) | small bump in chat, safety, truthfulness |
| + DPO on **UltraFeedback** (better, synthetic GPT-4-rated data) | bigger jumps than HH: **data matters most** |
| switch DPO → **PPO** (same data) | bump on more metrics (incl. factuality); biggest jump on AlpacaEval 2 |
| **scale up the reward model** | expected general gains; reality: challenging tasks like reasoning improve, others decline. **Training a good reward model is not easy.** |
| **add more prompts** to RLHF | expected broad gains; reality: some code/reasoning improvements, but messy |

**PPO takeaways:** *"Always one more thing to ablate."* *"PPO gets the best model, but we don't know why."* Generation is very slow without accelerated inference tools (e.g., vLLM): PPO samples from the policy at every step. Resources: they trained on TPUs (Google TPU Research Cloud) and could barely fit a 70B policy + 70B reward model on a 512-chip v3 pod. **The ordering of importance they found: data quality > algorithm (PPO > DPO) > reward model size > prompt quantity.**

---
## 6. Can we match PPO with "online" DPO?

**What is special about online data?** Online data is **freshly generated from the policy** and/or **recently labeled** by a reward model or judge. PPO does both (generation + RM scoring). Other methods get there differently: collecting new preference data, re-labeling existing data, LLM-as-a-judge, reward-model ranking. A related question is on-policy vs. off-policy data (whether the responses were generated by the model being trained).

**D2PO** (discriminator-guided DPO; Singhal et al. 2024) minimizes the "staleness" of DPO training data, by sampling new responses from the current policy and labeling them with a discriminator (reward model). With online methods, "DPO becomes horizontal lines" (all data used up) turns into learning curves that look much more like old-school RL. Industry does **both** online and **iterative** RLHF (Claude, Llama 2: sequential rounds of training and new preference collection); academia has mostly had a taste of the former.

### 6.1 Offline vs. online DPO, measured
The setting from L10: prompts with 10 candidate responses, a true quality per response, and an SFT/reference policy. The reference **rarely produces** some of the best responses. We compare:
* **Offline DPO:** one fixed dataset of preference pairs sampled from the reference policy.
* **Online (iterative) DPO:** in each round, sample fresh pairs **from the current policy**, label them with the judge, and take DPO steps. The same total number of labeled pairs.

In [ ]:
r = np.random.default_rng(3)
N_PROMPTS, K = 40, 10
q = r.normal(size=(N_PROMPTS, K))                                  # true quality
ref_logits = r.normal(0, 1.0, (N_PROMPTS, K))
best = q.argmax(1)
ref_logits[np.arange(N_PROMPTS), best] -= 3.0                      # the reference rarely produces the BEST response
p_ref = softmax(ref_logits)

def label(x, a, b, r_):                                            # judge: Bradley-Terry on true quality
    return (a, b) if r_.random() < sigmoid(3 * (q[x, a] - q[x, b])) else (b, a)

def sample_pairs(policy, n, r_):
    data = []
    for _ in range(n):
        x = int(r_.integers(N_PROMPTS)); a, b = r_.choice(K, 2, replace=True, p=policy[x])
        if a != b:
            data.append((x, *label(x, a, b, r_)))
    return np.array(data)

def dpo_steps(theta, data, beta=0.5, steps=200, lr=1.0):
    x, w, l = data[:, 0], data[:, 1], data[:, 2]
    for _ in range(steps):
        logp = theta - np.log(np.exp(theta).sum(1, keepdims=True))
        u = beta * ((logp[x, w] - np.log(p_ref[x, w])) - (logp[x, l] - np.log(p_ref[x, l])))
        coef = (1 - sigmoid(u)) * beta; grad = np.zeros_like(theta)
        np.add.at(grad, (x, w), coef); np.add.at(grad, (x, l), -coef)
        theta = theta + lr * grad / len(data) * N_PROMPTS
    return theta

TOTAL, ROUNDS = 4000, 8
offline_data = sample_pairs(p_ref, TOTAL, np.random.default_rng(10))
theta_off = dpo_steps(np.log(p_ref).copy(), offline_data, steps=800)
p_off = softmax(theta_off)

theta_on = np.log(p_ref).copy(); r_on = np.random.default_rng(10); curve = []
for round_ in range(ROUNDS):
    data = sample_pairs(softmax(theta_on), TOTAL // ROUNDS, r_on)     # fresh, on-policy pairs each round
    theta_on = dpo_steps(theta_on, data, steps=100)
    curve.append((softmax(theta_on) * q).sum(1).mean())
p_on = softmax(theta_on)

involves_best = np.mean((offline_data[:, 1] == best[offline_data[:, 0]]) | (offline_data[:, 2] == best[offline_data[:, 0]]))
print(f"offline dataset: {len(offline_data)} comparisons; only {involves_best:.1%} involve the best response for their prompt")
for name, pol in [("reference (SFT)", p_ref), ("offline DPO", p_off), ("online / iterative DPO", p_on)]:
    print(f"{name:24s} true quality {(pol * q).sum(1).mean():+.3f}   P(best response) {pol[np.arange(N_PROMPTS), best].mean():.3f}")
print("online DPO true quality after each round:", np.round(curve, 3))

Both improve on the reference, but **online DPO goes further with the same labeling budget.** The offline dataset was sampled from the reference model, so most of its comparisons are between responses the reference likes, and very few involve the best responses: the data is concentrated where the policy *was*, not where it's *going*. Online DPO keeps sampling from the *current* policy, so as the policy moves toward better responses, its new comparisons are about exactly the region it now cares about. That's the data-staleness argument behind D2PO, iterative DPO, and part of PPO's advantage. (A known further failure of offline DPO, not visible here because our 4,000 pairs cover nearly every response: with sparse data, probability can drift onto responses that never appear in the dataset, since DPO's gradient only touches the observed ones. Exercise 2b.)

---
## 7. Rejection sampling (best-of-n), and what Meta did with Llama 3

> *"Our approach to post-training is a combination of supervised fine-tuning (SFT), rejection sampling, proximal policy optimization (PPO), and direct preference optimization (DPO)."* (Llama 3)

Lambert's reading: **iterative data collection** (like Llama 2), **short timelines** for each iteration, some sort of "distribution shift" per method; hypothesis: **rejection sampling, then DPO, then PPO**.

**Rejection sampling / best-of-n:** sample $n$ responses, keep the one with the highest reward-model score, and use it as SFT data (or serve it directly). It's simple, stable, and parallel. It's also an RL-like policy improvement with a known KL from the reference policy: for continuous (all-distinct) samples, $\text{KL}(\pi_{\text{best-of-}n}\,\|\,\pi_{\text{ref}}) = \log n - \frac{n-1}{n}$ (Stiennon et al. 2020; Beirami et al. 2024 show it's an upper bound in general). Let's verify that, and watch over-optimization with an imperfect reward model:

In [ ]:
r = np.random.default_rng(0)
M = 200_000                                                       # a reference "policy" over many distinct responses
true_q = r.normal(size=M)
proxy = true_q + r.normal(0, 1.0, M) + 0.6 * np.maximum(r.normal(size=M), 0) ** 3 * (r.random(M) < 0.02)   # RM with rare big errors
ranks = np.argsort(np.argsort(proxy)) / M                          # quantile of each response under the proxy RM

print(f"{'n':>5s} {'KL formula':>11s} {'KL empirical':>13s} {'proxy reward':>13s} {'true quality':>13s}")
for n in [1, 2, 4, 16, 64, 256, 1024]:
    # best-of-n under the proxy: P(select a response with proxy quantile u) has density n u^(n-1)
    w = n * ranks ** (n - 1); w /= w.sum()
    kl_emp = np.sum(w * np.log(w * M + 1e-300))
    print(f"{n:5d} {math.log(n) - (n - 1) / n:11.3f} {kl_emp:13.3f} {np.sum(w * proxy):13.3f} {np.sum(w * true_q):13.3f}")

The KL matches the formula, and grows only logarithmically: best-of-1024 is just ~5.9 nats from the reference. Meanwhile the proxy reward keeps rising, but the true quality eventually stops tracking it, as best-of-n increasingly selects the rare responses the reward model **overrates**. That's over-optimization again (L10 §5.3), and why the **reward model's quality bounds everything downstream**.

---
## 8. Current directions, and life after *this* lecture

**Current directions (May 2024):**
1. **Data! Data! Data!** Experimentation is severely limited by having too few preference datasets (Anthropic HH, UltraFeedback, and Nectar are the main three).
2. **Continuing to improve DPO:** tons of papers iterating on the method (ORPO, cDPO, IPO, BCO, KTO, DNO, sDPO, …).
3. **More model sizes:** most alignment research happened at 7B or 13B; expand up and down!
4. **Specific evaluations:** how do we get evaluations more specific than Chatbot Arena?
5. **Personalization:** a large motivation behind local models; a young area academically.

**Where open alignment was happening:** AI2 (Tulu, OLMo), HuggingFace H4 (quick recipes, e.g., ORPO, CAI), Berkeley-Nest/Nexusflow (Nectar, Starling), NousResearch (Hermes), OpenBMB (preference data, RMs), Argilla (open preference data), and individual HF users (model merging, fine-tunes).

### 8.1 🔎 Beyond the slides: the DPO family, as loss functions
All of these take the policy/reference log-ratios $h_w = \log\frac{\pi(y_w)}{\pi_{ref}(y_w)}$, $h_l = \log\frac{\pi(y_l)}{\pi_{ref}(y_l)}$ (or variations):

| Method | Loss | Idea |
|---|---|---|
| **DPO** | $-\log\sigma(\beta(h_w - h_l))$ | Bradley–Terry on implicit rewards |
| **cDPO** | label-smoothed DPO: $-(1-\epsilon)\log\sigma(\beta\Delta) - \epsilon\log\sigma(-\beta\Delta)$ | robustness to flipped labels (§4.3) |
| **IPO** (Azar et al. 2023) | $\big(h_w - h_l - \frac{1}{2\beta}\big)^2$ | regress the margin to a target; doesn't push it to infinity on deterministic preferences |
| **KTO** (Ethayarajh et al. 2024) | per-example: desirable/undesirable labels, no pairs | uses binary "thumbs up/down" data |
| **ORPO** (Hong et al. 2024) | SFT loss $+\ \lambda\cdot(-\log\sigma(\log\text{odds}(y_w) - \log\text{odds}(y_l)))$ | no reference model; SFT and preference in one stage |
| **SimPO** (Meng et al. 2024) | $-\log\sigma\big(\frac{\beta}{|y_w|}\log\pi(y_w) - \frac{\beta}{|y_l|}\log\pi(y_l) - \gamma\big)$ | no reference model; **length-normalized** (counters length bias, cf. §4.2) |

In [ ]:
def dpo_loss(hw, hl, beta=0.1):
    return -np.log(sigmoid(beta * (hw - hl)))
def cdpo_loss(hw, hl, beta=0.1, eps=0.1):
    d = beta * (hw - hl); return -(1 - eps) * np.log(sigmoid(d)) - eps * np.log(sigmoid(-d))
def ipo_loss(hw, hl, beta=0.1):
    return (hw - hl - 1 / (2 * beta)) ** 2
def simpo_loss(logp_w, logp_l, len_w, len_l, beta=2.0, gamma=0.5):
    return -np.log(sigmoid(beta * logp_w / len_w - beta * logp_l / len_l - gamma))

margins = np.array([0.0, 5.0, 20.0, 100.0])               # h_w - h_l: how far the policy has already pushed the pair apart
print("margin h_w - h_l:", margins)
print("DPO  loss       :", dpo_loss(margins, 0).round(4), "  <- keeps decreasing: DPO pushes margins toward infinity")
print("cDPO loss       :", cdpo_loss(margins, 0).round(4), "  <- has a minimum: robust to label noise")
print("IPO  loss       :", ipo_loss(margins, 0).round(4), "  <- minimized at margin 1/(2*beta) = 5")
print("SimPO, a long chosen response (200 tokens, log-prob -300) vs. a short rejected one (50 tokens, -100):",
      round(float(simpo_loss(-300, -100, 200, 50)), 3), "(per-token log-probs compared, so length isn't penalized)")

### 8.2 🔎 Since this lecture: RL with verifiable rewards and GRPO
In late 2024 and 2025, the biggest change in post-training was the rise of **reinforcement learning with verifiable rewards (RLVR)** for math, code, and other checkable tasks. The term was popularized by AI2's Tulu 3 (Nov 2024), and the approach was used at scale by DeepSeek-R1 (Jan 2025) and other "reasoning" models. The reward is computed by a **program** (is the final answer correct? do the unit tests pass?), so there's no learned reward model to over-optimize in the way of §7.

A popular algorithm is **GRPO** (Group Relative Policy Optimization, from DeepSeekMath, Shao et al. 2024). For each prompt, sample a **group** of $G$ responses, score them, and use each response's reward **standardized within its group** as its advantage, with no value network (unlike PPO):
$$A_i = \frac{r_i - \text{mean}(r_{1..G})}{\text{std}(r_{1..G})},\qquad \nabla J \approx \frac1G\sum_i A_i\nabla\log\pi(y_i\mid x)\ \ (- \beta\nabla\text{KL})$$
(the full method also uses PPO-style ratio clipping). Groups where all responses are right, or all wrong, give zero advantage: the learning signal comes from prompts at the edge of the model's ability. A miniature version, on prompts where each of 10 candidate "solutions" is either correct or not:

In [ ]:
r = np.random.default_rng(0)
N_P, K_S = 50, 10
correct = r.random((N_P, K_S)) < 0.3                       # which candidate solutions are correct (the verifier knows)
correct[np.arange(N_P), r.integers(0, K_S, N_P)] = True    # every prompt has at least one correct solution
logits0 = r.normal(0, 1.0, (N_P, K_S)); theta = logits0.copy(); G, lr, beta_kl = 8, 0.5, 0.02
p0 = softmax(logits0)
history = []
for step in range(150):
    pol = softmax(theta); grad = np.zeros_like(theta); useful = 0
    for x in range(N_P):
        ys = r.choice(K_S, G, p=pol[x]); rewards = correct[x, ys].astype(float)      # verifiable reward: 1 if correct
        if rewards.std() == 0:
            continue                                                                  # all right or all wrong: no signal
        useful += 1
        A = (rewards - rewards.mean()) / rewards.std()
        for y_, a_ in zip(ys, A):
            grad[x] += a_ * (np.eye(K_S)[y_] - pol[x]) / G
    grad -= beta_kl * (pol * (np.log(pol) - np.log(p0)) - pol * (pol * (np.log(pol) - np.log(p0))).sum(1, keepdims=True))  # KL to the start
    theta += lr * grad
    if step % 30 == 0 or step == 149:
        history.append((step, (softmax(theta) * correct).sum(1).mean(), useful))
for step, acc, useful in history:
    print(f"step {step:3d}: expected accuracy {acc:.3f}   prompts with a useful (mixed) group: {useful}/{N_P}")

Accuracy climbs from the base model's level toward 1, and the number of prompts that still give a learning signal shrinks as the model gets them right consistently. That "no signal at 0% or 100%" property is why RLVR pipelines carefully choose prompts of the right difficulty.

**The big picture for an ML engineer (2025–26):** a typical post-training pipeline is **SFT** (instruction and chat data, often with distilled or synthetic data) → **preference optimization** (DPO-family or RLHF with a reward model, for helpfulness, style, safety) → **RL with verifiable rewards** (for math, code, reasoning), often iterated with fresh on-policy data, exactly the "online" lesson of §6.

---
## 9. Summary

| Topic | Takeaway |
|---|---|
| RLHF's role | Necessary but not sufficient for ChatGPT-like models; used by all major labs |
| IFT | Base model + chat template with special tokens; loss on assistant turns only |
| Open models 2023 | Alpaca/Vicuna (distillation, ShareGPT) → OASST1 (human data) → Zephyr β (DPO, lr ~5e-7) → Tulu 2 (DPO at 70B) → Starling (PPO, k-wise RM) |
| RewardBench | Accuracy on chosen/rejected pairs by category; Chat Hard is the discriminative one; safety must work in both directions |
| DPO as an RM | Needs the reference model; log-prob alone leaks the reference's (e.g., length) preferences |
| Label noise | ~70% human agreement caps measured RM accuracy |
| PPO vs. DPO | Data quality > algorithm (PPO ≳ DPO) > RM size > more prompts; PPO is slow and finicky |
| Online data | Fresh on-policy samples + fresh labels: offline DPO can't learn about responses the reference rarely produces |
| Best-of-n | KL = log n − (n−1)/n; simple, strong, but bounded by RM quality |
| 🔎 DPO variants | cDPO, IPO, KTO, ORPO, SimPO address label noise, over-separation, unpaired data, reference-free training, length |
| 🔎 RLVR / GRPO | Programmatic rewards + group-relative advantages; behind today's reasoning models |

### Exercises
1. In §4.1, design a reward model that passes all four categories *except* Chat Hard, and explain what feature it relies on.
2. In §6.1, vary the number of online rounds (1, 2, 4, 16) at a fixed total number of labeled pairs. Where's the sweet spot?
   2b. Shrink the offline dataset to 200 pairs and measure the probability offline DPO assigns to responses that never appear in it.
3. Replace the judge in §6.1 with an imperfect reward model (e.g., the length-biased one). Does online DPO now over-optimize?
4. Implement **KTO**'s loss for unpaired thumbs-up/down labels and run it in the §6.1 setting.
5. In §8.2, give partial credit (reward 0.5) for "almost correct" solutions. How does that change which prompts provide signal?
6. Read the Tulu 3 report and map each stage of its pipeline onto the sections of this notebook.

### References
* Lambert et al. 2024, *RewardBench*; Ivison et al. 2024, *Unpacking DPO and PPO*; Singhal et al. 2024 (D2PO).
* Rafailov et al. 2023 (DPO); Touvron et al. 2023 (Llama 2); Bai et al. 2022 (Constitutional AI); Llama 3 (Dubey et al. 2024).
* Tunstall et al. 2023 (Zephyr); Ivison et al. 2023 (Tulu 2); Köpf et al. 2023 (OpenAssistant); Cui et al. 2023 (UltraFeedback); Zhu et al. 2023 (Starling/Nectar).
* Röttger et al. 2023 (XSTest); Zeng et al. 2023 (LLMBar).
* Azar et al. 2023 (IPO); Ethayarajh et al. 2024 (KTO); Hong et al. 2024 (ORPO); Meng et al. 2024 (SimPO).
* Stiennon et al. 2020; Beirami et al. 2024 (best-of-n KL); Shao et al. 2024 (DeepSeekMath, GRPO); Lambert et al. 2024 (Tulu 3, RLVR); DeepSeek-AI 2025 (DeepSeek-R1).
* Lambert's blog, interconnects.ai, and *RLHF Book*.